In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8')
%matplotlib inline

In [4]:
# Reload the data — each notebook is self-contained so we repeat the loading steps
# This means the notebook works on its own without needing to run 01_eda first
df = pd.read_csv('../data/UK-HPI-full-file-2025-12.csv', low_memory=False)

# Filtering down to Yorkshire — using the regional aggregate rather than
# individual councils like Leeds or Sheffield, keeps the signal cleaner
region = 'Yorkshire and The Humber'
yorks = df[df['RegionName'] == region].copy()

# ONS stores dates as DD/MM/YYYY so need to tell pandas explicitly
yorks['Date'] = pd.to_datetime(yorks['Date'], format='%d/%m/%Y')
yorks = yorks.sort_values('Date').reset_index(drop=True)

# Pre-1995 data uses an older methodology — not worth including
yorks = yorks[yorks['Date'] >= '1995-01-01'].reset_index(drop=True)

# Only need the target variable for now, bringing in other columns later if needed
yorks = yorks[['Date', 'AveragePrice']]

print(f"Loaded {len(yorks)} rows")
print(f"Date range: {yorks['Date'].min().date()} → {yorks['Date'].max().date()}")

Loaded 372 rows
Date range: 1995-01-01 → 2025-12-01


In [ ]:
# Feature engineering — giving the model ways to understand where prices have been
# and what time of year it is. No external data needed, just the price history itself.
def build_features(df):
    df = df.copy()
    
    # Lag features — what was the price 1, 3, 6 and 12 months ago?
    # The model uses these to understand recent momentum and longer trends
    df['lag_1']  = df['AveragePrice'].shift(1)
    df['lag_3']  = df['AveragePrice'].shift(3)
    df['lag_6']  = df['AveragePrice'].shift(6)
    df['lag_12'] = df['AveragePrice'].shift(12)
    
    # Rolling averages — smoothed view of recent price trajectory
    # Helps the model ignore short-term noise and focus on the trend
    df['rolling_3']  = df['AveragePrice'].shift(1).rolling(3).mean()
    df['rolling_6']  = df['AveragePrice'].shift(1).rolling(6).mean()
    df['rolling_12'] = df['AveragePrice'].shift(1).rolling(12).mean()
    
    # Year-on-year % change — how fast are prices growing vs this time last year?
    df['yoy_pct'] = df['AveragePrice'].pct_change(12) * 100
    
    # Date features — captures the weak seasonality we saw in the decomposition
    df['month']   = df['Date'].dt.month
    df['quarter'] = df['Date'].dt.quarter
    df['year']    = df['Date'].dt.year
    
    return df

yorks = build_features(yorks)

# The lag and rolling features create NaN rows at the start — drop them
# We lose the first 12 rows but that's fine, plenty of data remaining
yorks = yorks.dropna().reset_index(drop=True)

print(f"Rows after feature engineering: {len(yorks)}")
print(f"Date range: {yorks['Date'].min().date()} → {yorks['Date'].max().date()}")
print(f"\nFeatures created:")
print([c for c in yorks.columns if c != 'AveragePrice'])
yorks.head()

In [ ]:
# Splitting into train and test sets
# Using 2023 onwards as the test set — that's about 24 months of unseen data
# Keeping it time-based rather than random, since shuffling time series data would be cheating
# (the model would be learning from the future to predict the past)
train = yorks[yorks['Date'] < '2023-01-01'].copy()
test  = yorks[yorks['Date'] >= '2023-01-01'].copy()

print(f"Train: {len(train)} rows ({train['Date'].min().date()} → {train['Date'].max().date()})")
print(f"Test:  {len(test)} rows ({test['Date'].min().date()} → {test['Date'].max().date()})")

# Define which columns are features vs target
# Dropping Date as the model can't use it directly — the date info is captured in month/quarter/year
features = ['lag_1', 'lag_3', 'lag_6', 'lag_12',
            'rolling_3', 'rolling_6', 'rolling_12',
            'yoy_pct', 'month', 'quarter', 'year']

target = 'AveragePrice'

X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Baseline — linear regression on all features
# This is our benchmark. If XGBoost can't beat this, it's not worth using.
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print(f"Linear Regression — Test Set Performance")
print(f"  MAE: £{mae_lr:,.0f}")
print(f"  R²:  {r2_lr:.4f}")

# Plot actual vs predicted so we can see where it goes wrong
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['Date'], y_test,     label='Actual',    linewidth=2, color='steelblue')
ax.plot(test['Date'], y_pred_lr,  label='Predicted', linewidth=2, color='tomato', linestyle='--')
ax.set_title('Linear Regression — Actual vs Predicted (2023–2025)')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from xgboost import XGBRegressor

# XGBoost — should handle the non-linear patterns and volatility better than linear regression
# n_estimators and learning_rate are starting defaults, we can tune later if needed
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb  = r2_score(y_test, y_pred_xgb)

print(f"XGBoost — Test Set Performance")
print(f"  MAE: £{mae_xgb:,.0f}")
print(f"  R²:  {r2_xgb:.4f}")
print(f"\nImprovement over linear regression:")
print(f"  MAE reduction: £{mae_lr - mae_xgb:,.0f}")

# Plot both models together so we can compare directly
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['Date'], y_test,        label='Actual',             linewidth=2, color='steelblue')
ax.plot(test['Date'], y_pred_lr,     label='Linear Regression',  linewidth=2, color='tomato',   linestyle='--')
ax.plot(test['Date'], y_pred_xgb,    label='XGBoost',            linewidth=2, color='green',     linestyle='--')
ax.set_title('Model Comparison — Actual vs Predicted (2023–2025)')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost can't extrapolate trends on its own — it only interpolates between
# values it's seen. Adding a numeric time index gives it an explicit trend signal.
yorks['time_index'] = np.arange(len(yorks))

# Rebuild train/test with the new feature
features = ['lag_1', 'lag_3', 'lag_6', 'lag_12',
            'rolling_3', 'rolling_6', 'rolling_12',
            'yoy_pct', 'month', 'quarter', 'year', 'time_index']

train = yorks[yorks['Date'] < '2023-01-01'].copy()
test  = yorks[yorks['Date'] >= '2023-01-01'].copy()

X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

xgb2 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb2.fit(X_train, y_train)
y_pred_xgb2 = xgb2.predict(X_test)

mae_xgb2 = mean_absolute_error(y_test, y_pred_xgb2)
r2_xgb2  = r2_score(y_test, y_pred_xgb2)

print(f"XGBoost v2 (with time index) — Test Set Performance")
print(f"  MAE: £{mae_xgb2:,.0f}")
print(f"  R²:  {r2_xgb2:.4f}")
print(f"\nvs Linear Regression: £{mae_lr:,.0f} MAE")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['Date'], y_test,         label='Actual',            linewidth=2, color='steelblue')
ax.plot(test['Date'], y_pred_lr,      label='Linear Regression', linewidth=2, color='tomato', linestyle='--')
ax.plot(test['Date'], y_pred_xgb2,    label='XGBoost v2',        linewidth=2, color='green',  linestyle='--')
ax.set_title('Model Comparison v2 — Actual vs Predicted (2023–2025)')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost struggles to extrapolate price levels beyond its training range
# Solution: predict month-on-month change instead, then reconstruct the price
# Changes are stationary (hover around 0) so XGBoost can handle them properly

yorks['price_change'] = yorks['AveragePrice'].diff()

# Rebuild with change as the target — drop the first row which is NaN
yorks_diff = yorks.dropna().reset_index(drop=True)

train_d = yorks_diff[yorks_diff['Date'] < '2023-01-01'].copy()
test_d  = yorks_diff[yorks_diff['Date'] >= '2023-01-01'].copy()

X_train_d = train_d[features]
y_train_d = train_d['price_change']
X_test_d  = test_d[features]
y_test_d  = test_d['price_change']

xgb3 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb3.fit(X_train_d, y_train_d)
predicted_changes = xgb3.predict(X_test_d)

# Reconstruct prices by adding predicted changes to the last known price
last_known_price = train_d['AveragePrice'].iloc[-1]
predicted_prices = [last_known_price]
for change in predicted_changes:
    predicted_prices.append(predicted_prices[-1] + change)
predicted_prices = predicted_prices[1:]  # drop the seed value

mae_xgb3 = mean_absolute_error(test_d['AveragePrice'], predicted_prices)
r2_xgb3  = r2_score(test_d['AveragePrice'], predicted_prices)

print(f"XGBoost v3 (predicting changes) — Test Set Performance")
print(f"  MAE: £{mae_xgb3:,.0f}")
print(f"  R²:  {r2_xgb3:.4f}")
print(f"\nvs Linear Regression: £{mae_lr:,.0f} MAE")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_d['Date'], test_d['AveragePrice'], label='Actual',            linewidth=2, color='steelblue')
ax.plot(test_d['Date'], y_pred_lr,              label='Linear Regression', linewidth=2, color='tomato', linestyle='--')
ax.plot(test_d['Date'], predicted_prices,       label='XGBoost v3',        linewidth=2, color='green',  linestyle='--')
ax.set_title('Model Comparison v3 — Actual vs Predicted (2023–2025)')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import QuantileRegressor

# Quantile regression — train three separate models for lower, mid and upper bounds
# This gives us a proper 90% confidence interval around each forecast
# q=0.05 → lower bound, q=0.50 → median forecast, q=0.95 → upper bound
quantiles = [0.05, 0.50, 0.95]
qr_models = {}
qr_preds  = {}

for q in quantiles:
    qr = QuantileRegressor(quantile=q, alpha=0, solver='highs')
    qr.fit(X_train, y_train)
    qr_models[q] = qr
    qr_preds[q]  = qr.predict(X_test)
    print(f"  q={q} trained — MAE: £{mean_absolute_error(y_test, qr_preds[q]):,.0f}")

# Plot the forecast with shaded confidence interval — this is the hero chart
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(test['Date'], y_test,          label='Actual',          linewidth=2,   color='steelblue')
ax.plot(test['Date'], qr_preds[0.50],  label='Median forecast', linewidth=2,   color='tomato', linestyle='--')
ax.fill_between(test['Date'],
                qr_preds[0.05],
                qr_preds[0.95],
                alpha=0.2, color='tomato', label='90% confidence interval')

ax.set_title('Yorkshire House Price Forecast with 90% Confidence Interval (2023–2025)')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Generate the actual forward forecasts — this is the headline output of the project
# We step forward one month at a time, using each predicted value as input for the next

def forecast_forward(models, last_row, n_months, features):
    """
    Rolls forward n_months from the last known data point.
    Each predicted price feeds back in as a lag feature for the next step.
    """
    current = last_row.copy()
    forecasts = {'date': [], 'p05': [], 'p50': [], 'p95': []}

    for i in range(1, n_months + 1):
        next_date  = current['Date'] + pd.DateOffset(months=1)
        input_df   = pd.DataFrame([current[features]])
        next_price = models[0.50].predict(input_df)[0]

        # Build next row by shifting lags forward
        new_row = current.copy()
        new_row['Date']         = next_date
        new_row['lag_12']       = current['lag_11'] if 'lag_11' in current else current['lag_12']
        new_row['lag_6']        = current['lag_5']  if 'lag_5'  in current else current['lag_6']
        new_row['lag_3']        = current['lag_2']  if 'lag_2'  in current else current['lag_3']
        new_row['lag_1']        = current['AveragePrice']
        new_row['AveragePrice'] = next_price
        new_row['month']        = next_date.month
        new_row['quarter']      = next_date.quarter
        new_row['year']         = next_date.year
        new_row['rolling_3']    = (current['AveragePrice'] + current['lag_1'] + current['lag_3']) / 3
        new_row['rolling_6']    = (current['AveragePrice'] + current['lag_1'] + current['lag_3'] + current['lag_6']) / 4
        new_row['rolling_12']   = (current['AveragePrice'] + current['lag_12']) / 2
        new_row['yoy_pct']      = ((next_price / current['lag_12']) - 1) * 100

        forecasts['date'].append(next_date)
        forecasts['p05'].append(models[0.05].predict(pd.DataFrame([current[features]]))[0])
        forecasts['p50'].append(next_price)
        forecasts['p95'].append(models[0.95].predict(pd.DataFrame([current[features]]))[0])

        current = new_row

    return pd.DataFrame(forecasts)

# Run the forecast from the last row of data
last_row    = yorks.iloc[-1].copy()
forecast_df = forecast_forward(qr_models, last_row, 12, features)

# Print the 6 and 12 month headline numbers
f6  = forecast_df.iloc[5]
f12 = forecast_df.iloc[11]

print("=" * 55)
print(f"6-month forecast  ({f6['date'].strftime('%b %Y')}):")
print(f"  Central:  £{f6['p50']:,.0f}")
print(f"  90% CI:   £{f6['p05']:,.0f} – £{f6['p95']:,.0f}")
print()
print(f"12-month forecast ({f12['date'].strftime('%b %Y')}):")
print(f"  Central:  £{f12['p50']:,.0f}")
print(f"  90% CI:   £{f12['p05']:,.0f} – £{f12['p95']:,.0f}")
print("=" * 55)

In [ ]:
# Final chart — historical prices + forward forecast with confidence band
# This is the second hero visual for the README
fig, ax = plt.subplots(figsize=(14, 6))

# Show last 3 years of history for context
history_window = yorks[yorks['Date'] >= '2023-01-01']
ax.plot(history_window['Date'], history_window['AveragePrice'],
        label='Historical', linewidth=2, color='steelblue')

# Forward forecast
ax.plot(forecast_df['date'], forecast_df['p50'],
        label='Forecast (median)', linewidth=2, color='tomato', linestyle='--')
ax.fill_between(forecast_df['date'], forecast_df['p05'], forecast_df['p95'],
                alpha=0.2, color='tomato', label='90% confidence interval')

# Mark the 6 and 12 month points explicitly
ax.axvline(f6['date'],  color='grey', linestyle=':', alpha=0.7)
ax.axvline(f12['date'], color='grey', linestyle=':', alpha=0.7)
ax.text(f6['date'],  ax.get_ylim()[0], ' 6m', fontsize=9, color='grey')
ax.text(f12['date'], ax.get_ylim()[0], ' 12m', fontsize=9, color='grey')

ax.set_title('Yorkshire House Price — 12 Month Forward Forecast')
ax.set_ylabel('Average Price (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()